[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/service_assistant/plain_python.ipynb)

# Simulated service assistant — plain Python

**Task.** Safely update one simulated order address using exact approval, checkpointing and idempotency. No external service is contacted.

**Read this notebook as:** choose a provider → load the simulation → declare safety boundaries → execute and evaluate.

In [ ]:
# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "service-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = "models/local/Qwen3-0.6B-Q8_0.gguf"
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.12.5"], check=True)

In [ ]:
import json
from pathlib import Path
from typing import ClassVar, Literal

from pydantic import BaseModel, ConfigDict

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import SimulatedService  # noqa: E402

service_path = ROOT / "data/service_assistant/simulated_service.json"
fixture_path = ROOT / "data/service_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Visible approval and recovery workflow
The model proposes an action but cannot execute it. Python checks action name, permission scope and an approval payload tied to every exact argument, checkpoints before the effect, then uses the idempotency key to make retries safe.

In [ ]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Action(Strict):
    schema_id: ClassVar[str] = "service.action.v1"
    action: Literal["update_address"]
    order_id: Literal["ord-100"]
    new_address: Literal["2 Evidence Road"]
    idempotency_key: Literal["address-ord-100-v1"]
    requires_approval: Literal[True]


class Confirmation(Strict):
    schema_id: ClassVar[str] = "service.confirmation.v1"
    message: str
    status: Literal["completed"]


Action.model_rebuild(_types_namespace={"Literal": Literal})
Confirmation.model_rebuild(_types_namespace={"Literal": Literal})


def model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


async def run_service():
    client = model()
    service = SimulatedService.from_path(service_path)
    trace = []
    before = service.read_order("ord-100", actor="tutorial-user")
    trace.append({"event": "read", "address": before["delivery_address"]})
    action = await propose(
        client,
        Action,
        "Propose update_address for order ord-100 to 2 Evidence Road, "
        "using idempotency key address-ord-100-v1 and requiring approval.",
    )
    allowed = action.action == "update_address" and action.requires_approval
    approval = {
        "order_id": "ord-100",
        "new_address": "2 Evidence Road",
        "idempotency_key": "address-ord-100-v1",
    }
    exact = approval == {
        "order_id": action.order_id,
        "new_address": action.new_address,
        "idempotency_key": action.idempotency_key,
    }
    if not allowed or not exact:
        return {"outcome": "refused", "trace": [*trace, {"event": "approval_denied"}]}
    checkpoint = service.checkpoint()
    trace.append({"event": "interrupted_after_checkpoint"})
    service = SimulatedService.resume(checkpoint)
    receipt = service.update_address(
        action.order_id,
        action.new_address,
        actor="tutorial-user",
        idempotency_key=action.idempotency_key,
    )
    duplicate = service.replay(action.idempotency_key)
    trace.extend(
        [
            {"event": "effect", "receipt": receipt},
            {"event": "duplicate_detected", "duplicate": duplicate["duplicate"]},
        ]
    )
    confirmation = await propose(
        client, Confirmation, f"Confirm this receipt only with status completed: {receipt}"
    )
    return {
        "outcome": confirmation.status,
        "receipt": receipt,
        "duplicate": duplicate,
        "trace": trace,
        "termination": "criteria_met",
        "usage": {"model_calls": 2, "tool_calls": 2},
    }


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_service()
second = await run_service() if MODEL_PROVIDER == "mock" else first
evaluation = {
    "component": first["receipt"]["delivery_address"] == "2 Evidence Road",
    "trajectory": len(first["trace"]) == 4,
    "task": first["outcome"] == "completed",
    "safety": first["duplicate"]["duplicate"] is True,
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "result": first,
    "resource_report": first["usage"],
    "fallback": "refuse or escalate when scope or exact approval differs",
}

## Limitations
The environment is intentionally simulated; real services require durable transactional storage, authenticated approvers and provider-specific failure handling.